# Week 2: Network Properties — Assignment

**Learning objectives** — In this assignment you will:

- Rank nodes by degree centrality
- Implement betweenness centrality from scratch (Brandes' algorithm)
- Implement local clustering coefficient from scratch
- Implement closeness centrality from scratch
- Measure the correlation between centrality measures using your own implementations

## Grading

| Section | Part | Function | Points |
|---------|------|----------|--------|
| 1 | Degree Centrality | `top_k_by_degree(G, k)` | 10 |
| 2 | Betweenness | `compute_betweenness(G, normalized)` | 15 |
| 3 | Clustering | `local_clustering(G, node)` | 20 |
| 4 | Closeness | `closeness_centrality(G, node)` | 10 |
| 5 | Correlation | `centrality_correlation(G)` | 15 |
| 6 | PageRank | `compute_pagerank(G, alpha, max_iter)` | 10 |
| 7 | Assortativity | `degree_assortativity(G)` | 10 |
| — | Written Questions | — | 10 |
| | **Total** | | **100** |

## Before You Start

This assignment builds on the Week 2 lab. Make sure you are comfortable with:

- **Degree distribution** — how to read linear and log-log plots (Lab Section 2)
- **Clustering coefficient** — the formula C = 2·triangles / k(k-1) and what it measures (Lab Section 4)
- **Three centrality measures** — degree (popularity), betweenness (brokerage), closeness (proximity) and how they can disagree (Lab Section 5)
- **PageRank** — recursive importance via random walks, damping factor α (Lab Section 6)
- **Assortativity** — do hubs connect to hubs? The degree correlation coefficient r (Lab Section 7)

You will implement **all centrality measures from scratch** — do not call `nx.degree_centrality`, `nx.betweenness_centrality`, `nx.closeness_centrality`, or `nx.pagerank` in your solutions. You may use basic graph primitives (`G.degree()`, `G.neighbors()`, `nx.shortest_path_length`) as building blocks.

**Important**: In Section 5, you must **reuse your own implementations** from earlier sections rather than calling NetworkX centrality functions.

In [1]:
import networkx as nx
import numpy as np
import matplotlib.pyplot as plt
from netsci.loaders import load_graph
from netsci.utils import SEED

In [2]:
G_karate = load_graph("karate")
G_fb = load_graph("facebook")

karate: 34 nodes, 78 edges (undirected)
facebook: 334 nodes, 2852 edges (undirected)


---
## Section 1: Degree Centrality (10 pts)

Return the top-*k* nodes by degree as a list of `(node, degree)` tuples, sorted from highest to lowest degree.

**Do not** call `nx.degree_centrality`. Use `G.degree()` to access raw degrees directly.

In [3]:
def top_k_by_degree(G, k=5):
    """Return the top-k nodes by degree.

    Do NOT call nx.degree_centrality.

    Parameters
    ----------
    G : nx.Graph
    k : int

    Returns
    -------
    list of (node, degree) tuples, sorted descending by degree.
    """
    # Get raw degrees and sort descending by degree.
    degs = list(G.degree())
    degs.sort(key=lambda x: x[1], reverse=True)
    return degs[:k]

In [4]:
# --- Validation ---
_top5 = top_k_by_degree(G_karate, 5)
assert len(_top5) == 5
# Degrees should be descending
assert all(_top5[i][1] >= _top5[i + 1][1] for i in range(len(_top5) - 1))
# Node 33 or 0 should be in top 5 (they are the highest-degree nodes)
_top_nodes = {n for n, d in _top5}
assert 33 in _top_nodes or 0 in _top_nodes
print(f"Top 5 in Karate: {_top5}")
print("Section 1 passed!")

Top 5 in Karate: [(33, 17), (0, 16), (32, 12), (2, 10), (1, 9)]
Section 1 passed!


---
## Section 2: Betweenness Centrality from Scratch (15 pts)

Implement betweenness centrality **from scratch** using Brandes' algorithm. **Do not** call `nx.betweenness_centrality`.

The algorithm works as follows — for each source node *s*:

1. **Forward phase (BFS)**: Run BFS from *s*. For each node *w*, track:
   - $d[w]$: shortest distance from *s*
   - $\sigma[w]$: number of shortest paths from *s* to *w*
   - $P[w]$: list of predecessors of *w* on shortest paths from *s*
2. **Back-propagation**: Process nodes in reverse BFS order. For each node *w* with predecessor *v*:
   - $\delta[v] \mathrel{+}= \frac{\sigma[v]}{\sigma[w]} \cdot (1 + \delta[w])$
   - Then add $\delta[w]$ to $\text{betweenness}[w]$ (for $w \neq s$)
3. **Correction**: For undirected graphs, divide all values by 2 (each pair is counted twice).
4. **Normalization**: If `normalized=True`, multiply by $\frac{2}{(n-1)(n-2)}$ for undirected graphs.

You may use `collections.deque` for the BFS queue.

In [5]:
def compute_betweenness(G, normalized=True):
    """Compute betweenness centrality for all nodes from scratch (Brandes' algorithm).

    Parameters
    ----------
    G : nx.Graph
    normalized : bool, default True
        If True, normalize by 2/((n-1)(n-2)) for undirected graphs.

    Returns
    -------
    dict mapping node -> float
    """
    from collections import deque

    nodes = list(G.nodes())
    n = len(nodes)
    betweenness = {v: 0.0 for v in nodes}

    for s in nodes:
        # Initialization
        stack = []
        preds = {v: [] for v in nodes}
        sigma = {v: 0.0 for v in nodes}
        dist = {v: -1 for v in nodes}

        sigma[s] = 1.0
        dist[s] = 0
        queue = deque([s])

        # Forward BFS
        while queue:
            v = queue.popleft()
            stack.append(v)
            for w in G.neighbors(v):
                # Path discovery
                if dist[w] < 0:
                    dist[w] = dist[v] + 1
                    queue.append(w)
                # Path counting
                if dist[w] == dist[v] + 1:
                    sigma[w] += sigma[v]
                    preds[w].append(v)

        # Back-propagation
        delta = {v: 0.0 for v in nodes}
        while stack:
            w = stack.pop()
            coeff = (1.0 + delta[w])
            for v in preds[w]:
                if sigma[w] > 0:
                    delta[v] += (sigma[v] / sigma[w]) * coeff
            if w != s:
                betweenness[w] += delta[w]

    # Correction for undirected graphs
    if not G.is_directed():
        for v in betweenness:
            betweenness[v] /= 2.0

        if normalized and n > 2:
            scale = 2.0 / ((n - 1) * (n - 2))
            for v in betweenness:
                betweenness[v] *= scale

    return betweenness

In [6]:
# --- Validation ---
_bet = compute_betweenness(G_karate, normalized=True)
_bet_nx = nx.betweenness_centrality(G_karate, normalized=True)
assert isinstance(_bet, dict)
assert len(_bet) == G_karate.number_of_nodes()
for node in G_karate.nodes():
    assert abs(_bet[node] - _bet_nx[node]) < 1e-6, f"Mismatch at node {node}"
print("Section 2 passed!")

Section 2 passed!


---
## Section 3: Local Clustering Coefficient (20 pts)

Implement the **local clustering coefficient** for a single node **from scratch** (do not call `nx.clustering`).

Recall: for a node $v$ with degree $k_v$, the clustering coefficient is:

$$C_v = \frac{2 \times |\text{edges among neighbors of } v|}{k_v (k_v - 1)}$$

If $k_v < 2$, return 0.0 (no triangle is possible).

In [7]:
def local_clustering(G, node):
    """Compute the local clustering coefficient of a node from scratch.

    Do NOT call nx.clustering.

    Parameters
    ----------
    G : nx.Graph
    node : any hashable

    Returns
    -------
    float
    """
    neighbors = list(G.neighbors(node))
    k = len(neighbors)
    if k < 2:
        return 0.0

    # Count edges among neighbors
    edges = 0
    neigh_set = set(neighbors)
    for u in neighbors:
        for v in G.neighbors(u):
            if v in neigh_set and v != u:
                edges += 1

    # Each edge counted twice in undirected graph
    edges = edges / 2
    return (2.0 * edges) / (k * (k - 1))

In [8]:
# --- Validation ---
for node in G_karate.nodes():
    _mine = local_clustering(G_karate, node)
    _nx_val = nx.clustering(G_karate, node)
    assert abs(_mine - _nx_val) < 1e-6, f"Node {node}: got {_mine}, expected {_nx_val}"
print("All 34 nodes match nx.clustering — Section 3 passed!")

All 34 nodes match nx.clustering — Section 3 passed!


---
## Section 4: Closeness Centrality (10 pts)

Implement **closeness centrality** for a single node **from scratch** (do not call `nx.closeness_centrality`).

$$C_C(v) = \frac{n - 1}{\sum_{u \neq v} d(v, u)}$$

where $d(v, u)$ is the shortest path length from $v$ to $u$, and $n$ is the number of nodes.

You may use `nx.shortest_path_length` to get distances.

In [9]:
def closeness_centrality(G, node):
    """Compute closeness centrality for a single node from scratch.

    Do NOT call nx.closeness_centrality.

    Parameters
    ----------
    G : nx.Graph
    node : any hashable

    Returns
    -------
    float
    """
    n = G.number_of_nodes()
    if n <= 1:
        return 0.0

    lengths = dict(nx.shortest_path_length(G, node))
    # lengths includes node itself with distance 0
    reachable = len(lengths)
    if reachable <= 1:
        return 0.0

    total_dist = sum(lengths.values())
    if total_dist <= 0:
        return 0.0

    # Match NetworkX behavior: scale by (reachable-1)/(n-1)
    return (reachable - 1) / (n - 1) * (reachable - 1) / total_dist

In [10]:
# --- Validation ---
for node in G_karate.nodes():
    _mine = closeness_centrality(G_karate, node)
    _nx_val = nx.closeness_centrality(G_karate, node)
    assert abs(_mine - _nx_val) < 1e-6, f"Node {node}: got {_mine}, expected {_nx_val}"
print("All 34 nodes match nx.closeness_centrality — Section 4 passed!")

All 34 nodes match nx.closeness_centrality — Section 4 passed!


---
## Section 5: Centrality Correlation (15 pts)

Compute the **Pearson correlation** between degree centrality and betweenness centrality across all nodes.

**Reuse your own implementations**: call your `compute_betweenness` from Section 2 and compute degree centrality as $C_D(v) = \frac{\deg(v)}{n - 1}$ (do not call `nx.degree_centrality` or `nx.betweenness_centrality`).

Use `np.corrcoef` to compute the Pearson coefficient. Return a single float.

In [11]:
def centrality_correlation(G):
    """Compute Pearson r between degree and betweenness centrality.

    Reuse your own compute_betweenness() from Section 2 and compute
    degree centrality as deg(v) / (n - 1).  Do NOT call nx.degree_centrality
    or nx.betweenness_centrality.

    Parameters
    ----------
    G : nx.Graph

    Returns
    -------
    float  (Pearson correlation coefficient)
    """
    n = G.number_of_nodes()
    if n <= 1:
        return 0.0

    betweenness = compute_betweenness(G, normalized=True)
    deg_centrality = {v: d / (n - 1) for v, d in G.degree()}

    nodes = list(G.nodes())
    deg_vals = np.array([deg_centrality[v] for v in nodes])
    bet_vals = np.array([betweenness[v] for v in nodes])

    # Pearson correlation
    return float(np.corrcoef(deg_vals, bet_vals)[0, 1])

In [12]:
# --- Validation ---
_r = centrality_correlation(G_karate)
assert isinstance(_r, float)
# Degree and betweenness are positively correlated in most real networks
assert _r > 0.3, f"Expected positive correlation, got {_r}"
assert _r <= 1.0

# Verify against direct computation
_deg = nx.degree_centrality(G_karate)
_bet = nx.betweenness_centrality(G_karate)
_nodes = list(G_karate.nodes())
_d = [_deg[n] for n in _nodes]
_b = [_bet[n] for n in _nodes]
_expected_r = float(np.corrcoef(_d, _b)[0, 1])
assert abs(_r - _expected_r) < 1e-6, f"Got {_r}, expected {_expected_r}"
print(f"Pearson r (degree vs betweenness) = {_r:.4f}")
print("Section 5 passed!")

Pearson r (degree vs betweenness) = 0.9146
Section 5 passed!


---
## Section 6: PageRank from Scratch (10 pts)

Implement PageRank using the **power iteration** method. **Do not** call `nx.pagerank`.

1. Initialize: every node gets score 1/n
2. Repeat for `max_iter` iterations:
   - For each node v: new_score(v) = (1 - alpha) / n + alpha * sum(score(u) * w(u,v) / weighted_degree(u) for u in neighbors of v)
3. Return the final scores as a dictionary

Use `G.degree(u, weight="weight")` for the weighted out-degree and `G[u][v].get("weight", 1)` for edge weights (some graphs have weighted edges).

The damping factor `alpha` (default 0.85) controls how likely the random surfer is to follow a link vs jump to a random page.

In [13]:
def compute_pagerank(G, alpha=0.85, max_iter=100):
    """Compute PageRank using power iteration.

    Do NOT call nx.pagerank.

    Parameters
    ----------
    G : nx.Graph
    alpha : float, default 0.85
        Damping factor.
    max_iter : int, default 100
        Number of power iterations.

    Returns
    -------
    dict mapping node -> float (PageRank score)
    """
    nodes = list(G.nodes())
    n = len(nodes)
    if n == 0:
        return {}

    # Initialize scores
    scores = {v: 1.0 / n for v in nodes}

    for _ in range(max_iter):
        new_scores = {v: (1.0 - alpha) / n for v in nodes}
        # Handle dangling nodes (zero weighted degree)
        dangling_sum = 0.0
        for u in nodes:
            wdeg = G.degree(u, weight="weight")
            if wdeg == 0:
                dangling_sum += scores[u]

        if dangling_sum > 0:
            add = alpha * dangling_sum / n
            for v in nodes:
                new_scores[v] += add

        for u in nodes:
            wdeg = G.degree(u, weight="weight")
            if wdeg == 0:
                continue
            for v in G.neighbors(u):
                weight = G[u][v].get("weight", 1)
                new_scores[v] += alpha * scores[u] * weight / wdeg

        # Normalize to sum to 1
        total = sum(new_scores.values())
        if total > 0:
            for v in new_scores:
                new_scores[v] /= total

        scores = new_scores

    return scores

In [14]:
# --- Validation ---
_pr = compute_pagerank(G_karate, alpha=0.85, max_iter=100)
_pr_nx = nx.pagerank(G_karate, alpha=0.85, max_iter=100)
assert isinstance(_pr, dict)
assert len(_pr) == G_karate.number_of_nodes()
# Check values are close to NetworkX (tolerance for convergence differences)
for node in G_karate.nodes():
    assert abs(_pr[node] - _pr_nx[node]) < 0.005, (
        f"Node {node}: got {_pr[node]:.6f}, expected {_pr_nx[node]:.6f}"
    )
# Sum should be approximately 1
assert abs(sum(_pr.values()) - 1.0) < 0.01, (
    f"Sum = {sum(_pr.values()):.4f}, expected ~1.0"
)
print(f"Top 3 by PageRank: {sorted(_pr, key=_pr.get, reverse=True)[:3]}")
print("Section 6 passed!")

Top 3 by PageRank: [33, 0, 32]
Section 6 passed!


---
## Section 7: Degree Assortativity from Scratch (10 pts)

Implement the **degree assortativity coefficient** from scratch using Pearson correlation of degrees at edge endpoints. **Do not** call `nx.degree_assortativity_coefficient`.

For each edge (u, v) in an undirected graph, include **both** orderings: (deg(u), deg(v)) and (deg(v), deg(u)). This symmetrization matches Newman's original formula. Then compute the Pearson correlation coefficient between the two degree sequences using `np.corrcoef`.

In [15]:
def degree_assortativity(G):
    """Compute degree assortativity coefficient from scratch.

    Do NOT call nx.degree_assortativity_coefficient.

    Parameters
    ----------
    G : nx.Graph

    Returns
    -------
    float (Pearson correlation of degrees at edge endpoints)
    """
    deg = dict(G.degree())
    x = []
    y = []

    for u, v in G.edges():
        x.append(deg[u])
        y.append(deg[v])
        x.append(deg[v])
        y.append(deg[u])

    if len(x) == 0:
        return 0.0

    r = float(np.corrcoef(x, y)[0, 1])
    if np.isnan(r):
        return 0.0
    return r

In [16]:
# --- Validation ---
_r = degree_assortativity(G_karate)
_r_nx = nx.degree_assortativity_coefficient(G_karate)
assert isinstance(_r, float)
assert abs(_r - _r_nx) < 0.05, f"Got {_r:.4f}, expected {_r_nx:.4f}"

# Test on Facebook too
_r_fb = degree_assortativity(G_fb)
_r_fb_nx = nx.degree_assortativity_coefficient(G_fb)
assert abs(_r_fb - _r_fb_nx) < 0.05, (
    f"Facebook: got {_r_fb:.4f}, expected {_r_fb_nx:.4f}"
)

print(f"Karate:   r = {_r:.4f} (nx: {_r_nx:.4f})")
print(f"Facebook: r = {_r_fb:.4f} (nx: {_r_fb_nx:.4f})")
print("Section 7 passed!")

Karate:   r = -0.4756 (nx: -0.4756)
Facebook: r = -0.1371 (nx: -0.1371)
Section 7 passed!


---
## Written Questions (10 pts)

### Question 1 (5 pts)

In the US power grid, what does it mean for a node to have high **betweenness centrality**?
What would happen to the network if that node (substation) failed?

*Hints to guide your thinking:*
- *Betweenness counts how many shortest paths pass **through** a node. In a sparse, tree-like network like the power grid, what happens when a node on the "trunk" is removed?*
- *Think about the difference between a node with many local connections (high degree) vs. a node that sits on the only path between two regions (high betweenness).*
- *Consider real infrastructure: if a key highway interchange fails, what happens to traffic flow?*

**Your Answer:**

A high-betweenness substation sits on many of the shortest paths that connect different regions of the grid. In a sparse, tree-like power network, that means it is a "bridge" or trunk node: most power routes between groups of substations pass through it. If it fails, large parts of the grid can become disconnected or must reroute power along much longer (and less efficient) paths, increasing stress on other lines and raising the risk of cascading outages.


### Question 2 (5 pts)

Can a node have **high degree** but **low betweenness**? Describe a network structure where this happens and explain why.

*Hints to guide your thinking:*
- *Imagine a "star" subgraph: one central node connected to 50 leaf nodes, all of whom also connect to a separate hub. The central node has high degree — but do shortest paths between other nodes need to pass through it?*
- *Betweenness is about being on shortest paths **between other pairs**. A node can be popular (many friends) yet replaceable (all its friends also know each other directly).*
- *Think about the Karate Club: node 33 has the highest degree (17) — does it also have the highest betweenness? Check and explain why or why not.*

**Your Answer:**

Yes — a node can have high degree but low betweenness when it is well connected locally but not critical for connecting other pairs of nodes. For example, a node in a dense community (or a star where all leaves also connect to another hub) can have many neighbors, yet most shortest paths between other nodes bypass it because those neighbors are directly connected to each other. In the Karate Club graph, node 33 has the highest degree but does not necessarily have the highest betweenness because other nodes provide alternate routes between groups, so node 33 is not on the majority of shortest paths between other pairs.
